# Lecture 3 — Improving the Resume Scorer

In this notebook we score resumes against a job description and submit results to a shared leaderboard.

**Gold** resumes (prefixed `g`) are strong matches for the role. **Silver** resumes (prefixed `s`) are weaker. A good scoring strategy should clearly separate gold from silver.

In [ ]:
import random
from pydantic import BaseModel, Field
from typing import List

from resume_utils import load_resumes, load_job_requirements, analyze_resume, submit_score

OPENROUTER_API_KEY = ""  # <-- Paste your OpenRouter API key here
TEAM_NAME = ""    # <-- Change this to your team name

# YOU CAN USE ANY MODEL WITH STRUCTURED OUTPUTS:
# https://openrouter.ai/models?fmt=cards&supported_parameters=structured_outputs
# MODEL = "google/gemma-4-26b-a4b-it"
# MODEL= "google/gemma-3n-e4b-it"
# MODEL= "mistralai/mistral-small-2603"
MODEL="meta-llama/llama-3.1-8b-instruct"

if TEAM_NAME.strip() == "" or OPENROUTER_API_KEY.strip() == "":
    raise Exception("Missing Parameter TEAM_NAME or OPENROUTER_API_KEY")


all_resumes = load_resumes('../data/resumes_final_with_gold_silver.csv')
job_req = load_job_requirements('../data/job_req_senior.md')

# Score gold + silver + a small sample of wild resumes
gold   = {rid: r for rid, r in all_resumes.items() if rid.startswith('g')}
silver = {rid: r for rid, r in all_resumes.items() if rid.startswith('s')}
wild   = {rid: r for rid, r in all_resumes.items() if rid[0].isdigit()}

random.seed(42)
wild_sample = dict(random.sample(list(wild.items()), 4))

resumes = {**gold, **silver, **wild_sample}
print(f"Model: {MODEL}")
print(f"Scoring {len(resumes)} resumes: {len(gold)} gold, {len(silver)} silver, {len(wild_sample)} wild")

## Single Shot Prompt

In [ ]:
class ResumeScore(BaseModel):
    score: int = Field(..., ge=0, le=100, description="Resume match score from 0 to 100")
    reasoning: str = Field(..., description="Brief explanation of the score")

prompt = f"""Score this resume against the job requirements on a 0-100 scale.
    Provide a score and brief reasoning.

    Job Requirements: {job_req}"""

scores = {}
total_cost = 0.0
for i, (rid, resume) in enumerate(sorted(resumes.items())):
    print(f"  {i+1}/{len(resumes)}: Resume ID: {rid}", end=" ")
    resp = analyze_resume(
        api_key=OPENROUTER_API_KEY,
        prompt=prompt,
        resume_text=resume['Resume_str'],
        output_schema=ResumeScore,
        model=MODEL,
    )
    cost = resp.get('cost')
    if cost:
        total_cost += cost

    if resp['result']:
        scores[rid] = resp['result']['score']
        print(f"-> {scores[rid]}" + (f" (${cost:.5f})" if cost else ""))
    else:
        print(f"-> ERROR: {resp['error']}")
        scores[rid] = 0

    submit_score(TEAM_NAME, rid, scores[rid], cost=cost)

print(f"\nDone. Submitted {len(scores)} scores for team '{TEAM_NAME}'. Total cost: ${total_cost:.5f}")

## Multi-Stage Prompt

Instead of asking the model to produce a single score in one shot, we decompose the evaluation into focused extraction steps and then combine the results in code.

In [ ]:
# --- Stage 1: Extract experience ---
class ExperienceExtract(BaseModel):
    years_relevant: int = Field(..., ge=0, le=50, description="Years of experience relevant to the job requirements")
    evidence: str = Field(..., description="Exact quote(s) from the resume supporting the years claim")

# --- Stage 2: Extract skills match ---
class SkillsExtract(BaseModel):
    matched_skills: List[str] = Field(..., description="Skills from the job requirements found in the resume")
    missing_skills: List[str] = Field(..., description="Skills from the job requirements NOT found in the resume")
    evidence: str = Field(..., description="Exact quote(s) from the resume supporting matched skills")

experience_prompt = f"""Extract the candidate's years of relevant experience for this role.
Only count experience directly relevant to the job requirements.
You MUST cite exact text from the resume as evidence.

Job Requirements:
{job_req}"""

skills_prompt = f"""Compare the candidate's skills against the job requirements.
List which required skills are present and which are missing.
You MUST cite exact text from the resume as evidence for matched skills.

Job Requirements:
{job_req}"""

# --- Stage 3: Combine in code ---
def compute_composite_score(experience: dict, skills: dict) -> int:
    """Combine extraction results into a final score using deterministic logic."""
    score = 0

    # Experience component (0-40 points)
    yrs = experience.get("years_relevant", 0)
    if yrs >= 7:
        score += 40
    elif yrs >= 4:
        score += 25
    elif yrs >= 2:
        score += 15
    else:
        score += 5

    # Skills component (0-60 points)
    matched = len(skills.get("matched_skills", []))
    missing = len(skills.get("missing_skills", []))
    total = matched + missing
    if total > 0:
        score += int(60 * matched / total)

    return min(score, 100)

# --- Run all stages on each resume ---
multi_scores = {}
total_cost = 0.0
for i, (rid, resume) in enumerate(sorted(resumes.items())):
    print(f"  {i+1}/{len(resumes)}: Resume ID: {rid}")
    resume_cost = 0.0

    # Stage 1: Experience
    exp_resp = analyze_resume(
        api_key=OPENROUTER_API_KEY,
        prompt=experience_prompt,
        resume_text=resume['Resume_str'],
        output_schema=ExperienceExtract,
        model=MODEL,
    )
    exp_result = exp_resp['result'] or {"years_relevant": 0, "evidence": ""}
    if exp_resp.get('cost'):
        resume_cost += exp_resp['cost']

    # Stage 2: Skills
    skills_resp = analyze_resume(
        api_key=OPENROUTER_API_KEY,
        prompt=skills_prompt,
        resume_text=resume['Resume_str'],
        output_schema=SkillsExtract,
        model=MODEL,
    )
    skills_result = skills_resp['result'] or {"matched_skills": [], "missing_skills": [], "evidence": ""}
    if skills_resp.get('cost'):
        resume_cost += skills_resp['cost']

    # Stage 3: Combine
    final_score = compute_composite_score(exp_result, skills_result)
    multi_scores[rid] = final_score
    total_cost += resume_cost

    matched = len(skills_result.get("matched_skills", []))
    missing = len(skills_result.get("missing_skills", []))
    cost_str = f" | Cost: ${resume_cost:.5f}" if resume_cost else ""
    print(f"    Experience: {exp_result.get('years_relevant', '?')}yr | Skills: {matched}/{matched+missing} matched | Score: {final_score}{cost_str}")

    submit_score(TEAM_NAME, rid, final_score, cost=resume_cost if resume_cost else None)

print(f"\nDone. Submitted {len(multi_scores)} multi-stage scores for team '{TEAM_NAME}'. Total cost: ${total_cost:.5f}")